<a href="https://colab.research.google.com/github/TripuraMetta/AV.SC.U4AIE23125/blob/main/notebooks/unit8/unit8_part1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Unit 8: Proximal Policy Gradient (PPO) with PyTorch 🤖

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit9/thumbnail.png" alt="Unit 8"/>


In this notebook, you'll learn to **code your PPO agent from scratch with PyTorch using CleanRL implementation as model**.

To test its robustness, we're going to train it in:

- [LunarLander-v2 🚀](https://www.gymlibrary.dev/environments/box2d/lunar_lander/)


⬇️ Here is an example of what you will achieve. ⬇️

In [ ]:
%%html
<video controls autoplay><source src="https://huggingface.co/sb3/ppo-LunarLander-v2/resolve/main/replay.mp4" type="video/mp4"></video>

We're constantly trying to improve our tutorials, so **if you find some issues in this notebook**, please [open an issue on the GitHub Repo](https://github.com/huggingface/deep-rl-class/issues).

## Objectives of this notebook 🏆

At the end of the notebook, you will:

- Be able to **code your PPO agent from scratch using PyTorch**.
- Be able to **push your trained agent and the code to the Hub** with a nice video replay and an evaluation score 🔥.




## This notebook is from the Deep Reinforcement Learning Course
<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/deep-rl-course-illustration.jpg" alt="Deep RL Course illustration"/>

In this free course, you will:

- 📖 Study Deep Reinforcement Learning in **theory and practice**.
- 🧑‍💻 Learn to **use famous Deep RL libraries** such as Stable Baselines3, RL Baselines3 Zoo, CleanRL and Sample Factory 2.0.
- 🤖 Train **agents in unique environments**

Don’t forget to **<a href="http://eepurl.com/ic5ZUD">sign up to the course</a>** (we are collecting your email to be able to **send you the links when each Unit is published and give you information about the challenges and updates).**


The best way to keep in touch is to join our discord server to exchange with the community and with us 👉🏻 https://discord.gg/ydHrjt3WP5

## Prerequisites 🏗️
Before diving into the notebook, you need to:

🔲 📚 Study [PPO by reading Unit 8](https://huggingface.co/deep-rl-course/unit8/introduction) 🤗  

To validate this hands-on for the [certification process](https://huggingface.co/deep-rl-course/en/unit0/introduction#certification-process), you need to push one model, we don't ask for a minimal result but we **advise you to try different hyperparameters settings to get better results**.

If you don't find your model, **go to the bottom of the page and click on the refresh button**

For more information about the certification process, check this section 👉 https://huggingface.co/deep-rl-course/en/unit0/introduction#certification-process

## Set the GPU 💪
- To **accelerate the agent's training, we'll use a GPU**. To do that, go to `Runtime > Change Runtime type`

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/gpu-step1.jpg" alt="GPU Step 1">

- `Hardware Accelerator > GPU`

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/gpu-step2.jpg" alt="GPU Step 2">

## Create a virtual display 🔽

During the notebook, we'll need to generate a replay video. To do so, with colab, **we need to have a virtual screen to be able to render the environment** (and thus record the frames).

Hence the following cell will install the librairies and create and run a virtual screen 🖥

In [ ]:
!pip install setuptools==65.5.0

In [ ]:
%%capture
!apt install python-opengl
!apt install ffmpeg
!apt install xvfb
!apt install swig cmake
!pip install pyglet==1.5
!pip3 install pyvirtualdisplay

In [ ]:
# Virtual display
from pyvirtualdisplay import Display

virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

## Install dependencies 🔽
For this exercise, we use `gym==0.22`.

In [ ]:
!pip install swig==4.1.1

In [ ]:
!pip install box2d-py==2.3.5 --no-build-isolation

In [ ]:
!pip install gym==0.22
!pip install imageio-ffmpeg
!pip install huggingface_hub
!pip install gym[box2d]==0.22

In [ ]:
!pip install Box2D==2.3.10

In [ ]:
import Box2D

print("Box2D imported successfully!")

In [ ]:
import setuptools
import gym

print("Setuptools:", setuptools.__version__)
print("Gym:", gym.__version__)

In [ ]:
env = gym.make("LunarLander-v2")

print("LunarLander created successfully!")
print("Observation space:", env.observation_space)
print("Action space:", env.action_space)

In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## Let's code PPO from scratch with Costa Huang tutorial
- For the core implementation of PPO we're going to use the excellent [Costa Huang](https://costa.sh/) tutorial.
- In addition to the tutorial, to go deeper you can read the 37 core implementation details: https://iclr-blog-track.github.io/2022/03/25/ppo-implementation-details/

👉 The video tutorial: https://youtu.be/MEt6rrxH8W4

In [ ]:
from IPython.display import HTML

HTML('<iframe width="560" height="315" src="https://www.youtube.com/embed/MEt6rrxH8W4" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture" allowfullscreen></iframe>')

- The best is to code first on the cell below, this way, if you kill the machine **you don't loose the implementation**.

In [ ]:
### Your code here:
import os
import random
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import gym

from torch.distributions.categorical import Categorical
from torch.utils.tensorboard import SummaryWriter


# =========================
# PPO CONFIGURATION
# =========================

class Args:
    exp_name = "ppo-LunarLander-v2"
    seed = 1
    torch_deterministic = True
    cuda = True

    env_id = "LunarLander-v2"
    total_timesteps = 50000

    learning_rate = 2.5e-4
    num_envs = 4
    num_steps = 128

    anneal_lr = True
    gae = True
    gamma = 0.99
    gae_lambda = 0.95

    num_minibatches = 4
    update_epochs = 4

    norm_adv = True
    clip_coef = 0.2
    clip_vloss = True

    ent_coef = 0.01
    vf_coef = 0.5
    max_grad_norm = 0.5
    target_kl = None

    repo_id = "Tripura8928/ppo-LunarLander-v2"


args = Args()

args.batch_size = args.num_envs * args.num_steps
args.minibatch_size = args.batch_size // args.num_minibatches


# =========================
# DEVICE
# =========================

device = torch.device(
    "cuda" if torch.cuda.is_available() and args.cuda else "cpu"
)

print("Using device:", device)


# =========================
# SEED
# =========================

random.seed(args.seed)
np.random.seed(args.seed)
torch.manual_seed(args.seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(args.seed)


# =========================
# ENVIRONMENT
# =========================

def make_env(env_id, seed):
    env = gym.make(env_id)
    env.seed(seed)
    return env


envs = [make_env(args.env_id, args.seed + i) for i in range(args.num_envs)]


# =========================
# PPO AGENT
# =========================

class Agent(nn.Module):

    def __init__(self, envs):
        super().__init__()

        obs_shape = envs[0].observation_space.shape
        action_dim = envs[0].action_space.n

        self.critic = nn.Sequential(
            nn.Linear(np.prod(obs_shape), 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, 1),
        )

        self.actor = nn.Sequential(
            nn.Linear(np.prod(obs_shape), 64),
            nn.Tanh(),
            nn.Linear(64, 64),
            nn.Tanh(),
            nn.Linear(64, action_dim),
        )

    def get_value(self, x):
        return self.critic(x)

    def get_action_and_value(self, x, action=None):

        logits = self.actor(x)

        probs = Categorical(logits=logits)

        if action is None:
            action = probs.sample()

        return (
            action,
            probs.log_prob(action),
            probs.entropy(),
            self.critic(x),
        )


agent = Agent(envs).to(device)

optimizer = optim.Adam(
    agent.parameters(),
    lr=args.learning_rate,
    eps=1e-5,
)

print("PPO agent created successfully!")
print("Parameters:", sum(p.numel() for p in agent.parameters()))

In [ ]:
# =========================
# PPO TRAINING SETUP
# =========================

from types import SimpleNamespace

# Convert our configuration into an argparse-like object
args = SimpleNamespace(
    exp_name="ppo-LunarLander-v2",
    seed=1,
    torch_deterministic=True,
    cuda=True,
    env_id="LunarLander-v2",
    total_timesteps=50000,
    learning_rate=2.5e-4,
    num_envs=4,
    num_steps=128,
    anneal_lr=True,
    gae=True,
    gamma=0.99,
    gae_lambda=0.95,
    num_minibatches=4,
    update_epochs=4,
    norm_adv=True,
    clip_coef=0.2,
    clip_vloss=True,
    ent_coef=0.01,
    vf_coef=0.5,
    max_grad_norm=0.5,
    target_kl=None,
    repo_id="Tripura8928/ppo-LunarLander-v2",
)

args.batch_size = args.num_envs * args.num_steps
args.minibatch_size = args.batch_size // args.num_minibatches

run_name = f"{args.env_id}__{args.exp_name}__{args.seed}__{int(time.time())}"

print("Environment:", args.env_id)
print("Parallel environments:", args.num_envs)
print("Steps per rollout:", args.num_steps)
print("Batch size:", args.batch_size)
print("Minibatch size:", args.minibatch_size)
print("Total timesteps:", args.total_timesteps)
print("Run name:", run_name)

In [ ]:
# ============================================================
# PPO IMPLEMENTATION - UNIT 8
# Based on the official Hugging Face / CleanRL implementation
# ============================================================

import argparse
import os
import random
import time

import gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions.categorical import Categorical
from torch.utils.tensorboard import SummaryWriter


# ------------------------------------------------------------
# 1. ARGUMENTS
# ------------------------------------------------------------

def strtobool(value):
    value = str(value).lower()
    if value in ("y", "yes", "t", "true", "1"):
        return True
    if value in ("n", "no", "f", "false", "0"):
        return False
    raise ValueError(f"Invalid boolean value: {value}")


def parse_args():
    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--exp-name",
        type=str,
        default="ppo-LunarLander-v2",
    )

    parser.add_argument(
        "--seed",
        type=int,
        default=1,
    )

    parser.add_argument(
        "--torch-deterministic",
        type=lambda x: strtobool(x),
        default=True,
        nargs="?",
        const=True,
    )

    parser.add_argument(
        "--cuda",
        type=lambda x: strtobool(x),
        default=True,
        nargs="?",
        const=True,
    )

    parser.add_argument(
        "--track",
        type=lambda x: strtobool(x),
        default=False,
        nargs="?",
        const=True,
    )

    parser.add_argument(
        "--wandb-project-name",
        type=str,
        default="cleanRL",
    )

    parser.add_argument(
        "--wandb-entity",
        type=str,
        default=None,
    )

    parser.add_argument(
        "--capture-video",
        type=lambda x: strtobool(x),
        default=False,
        nargs="?",
        const=True,
    )

    # PPO parameters
    parser.add_argument(
        "--env-id",
        type=str,
        default="LunarLander-v2",
    )

    parser.add_argument(
        "--total-timesteps",
        type=int,
        default=50000,
    )

    parser.add_argument(
        "--learning-rate",
        type=float,
        default=2.5e-4,
    )

    parser.add_argument(
        "--num-envs",
        type=int,
        default=4,
    )

    parser.add_argument(
        "--num-steps",
        type=int,
        default=128,
    )

    parser.add_argument(
        "--anneal-lr",
        type=lambda x: strtobool(x),
        default=True,
        nargs="?",
        const=True,
    )

    parser.add_argument(
        "--gae",
        type=lambda x: strtobool(x),
        default=True,
        nargs="?",
        const=True,
    )

    parser.add_argument(
        "--gamma",
        type=float,
        default=0.99,
    )

    parser.add_argument(
        "--gae-lambda",
        type=float,
        default=0.95,
    )

    parser.add_argument(
        "--num-minibatches",
        type=int,
        default=4,
    )

    parser.add_argument(
        "--update-epochs",
        type=int,
        default=4,
    )

    parser.add_argument(
        "--norm-adv",
        type=lambda x: strtobool(x),
        default=True,
        nargs="?",
        const=True,
    )

    parser.add_argument(
        "--clip-coef",
        type=float,
        default=0.2,
    )

    parser.add_argument(
        "--clip-vloss",
        type=lambda x: strtobool(x),
        default=True,
        nargs="?",
        const=True,
    )

    parser.add_argument(
        "--ent-coef",
        type=float,
        default=0.01,
    )

    parser.add_argument(
        "--vf-coef",
        type=float,
        default=0.5,
    )

    parser.add_argument(
        "--max-grad-norm",
        type=float,
        default=0.5,
    )

    parser.add_argument(
        "--target-kl",
        type=float,
        default=None,
    )

    # Hugging Face
    parser.add_argument(
        "--repo-id",
        type=str,
        default="Tripura8928/ppo-LunarLander-v2",
    )

    # IMPORTANT: [] prevents Colab's own command-line arguments
    # from interfering with argparse.
    args = parser.parse_args(args=[])

    args.batch_size = args.num_envs * args.num_steps
    args.minibatch_size = args.batch_size // args.num_minibatches

    return args


# ------------------------------------------------------------
# 2. ENVIRONMENT
# ------------------------------------------------------------

def make_env(env_id, seed, idx, capture_video, run_name):

    def thunk():

        env = gym.make(env_id)

        env = gym.wrappers.RecordEpisodeStatistics(env)

        if capture_video and idx == 0:
            env = gym.wrappers.RecordVideo(
                env,
                f"videos/{run_name}"
            )

        env.seed(seed)
        env.action_space.seed(seed)
        env.observation_space.seed(seed)

        return env

    return thunk


# ------------------------------------------------------------
# 3. NETWORK INITIALIZATION
# ------------------------------------------------------------

def layer_init(
    layer,
    std=np.sqrt(2),
    bias_const=0.0
):
    torch.nn.init.orthogonal_(
        layer.weight,
        std
    )

    torch.nn.init.constant_(
        layer.bias,
        bias_const
    )

    return layer


# ------------------------------------------------------------
# 4. PPO AGENT
# ------------------------------------------------------------

class Agent(nn.Module):

    def __init__(self, envs):

        super().__init__()

        self.critic = nn.Sequential(

            layer_init(
                nn.Linear(
                    np.array(
                        envs.single_observation_space.shape
                    ).prod(),
                    64
                )
            ),

            nn.Tanh(),

            layer_init(
                nn.Linear(64, 64)
            ),

            nn.Tanh(),

            layer_init(
                nn.Linear(64, 1),
                std=1.0
            ),
        )

        self.actor = nn.Sequential(

            layer_init(
                nn.Linear(
                    np.array(
                        envs.single_observation_space.shape
                    ).prod(),
                    64
                )
            ),

            nn.Tanh(),

            layer_init(
                nn.Linear(64, 64)
            ),

            nn.Tanh(),

            layer_init(
                nn.Linear(
                    64,
                    envs.single_action_space.n
                ),
                std=0.01
            ),
        )

    def get_value(self, x):
        return self.critic(x)

    def get_action_and_value(
        self,
        x,
        action=None
    ):

        logits = self.actor(x)

        probs = Categorical(
            logits=logits
        )

        if action is None:
            action = probs.sample()

        return (
            action,
            probs.log_prob(action),
            probs.entropy(),
            self.critic(x),
        )


# ------------------------------------------------------------
# 5. START PPO
# ------------------------------------------------------------

args = parse_args()

run_name = (
    f"{args.env_id}__"
    f"{args.exp_name}__"
    f"{args.seed}__"
    f"{int(time.time())}"
)

print("=" * 60)
print("PPO TRAINING")
print("=" * 60)
print("Environment:", args.env_id)
print("Total timesteps:", args.total_timesteps)
print("Parallel environments:", args.num_envs)
print("Steps per rollout:", args.num_steps)
print("Batch size:", args.batch_size)
print("Minibatch size:", args.minibatch_size)
print("Repository:", args.repo_id)


# ------------------------------------------------------------
# 6. SEEDING
# ------------------------------------------------------------

random.seed(args.seed)
np.random.seed(args.seed)
torch.manual_seed(args.seed)

torch.backends.cudnn.deterministic = (
    args.torch_deterministic
)


# ------------------------------------------------------------
# 7. DEVICE
# ------------------------------------------------------------

device = torch.device(
    "cuda"
    if torch.cuda.is_available() and args.cuda
    else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


# ------------------------------------------------------------
# 8. VECTOR ENVIRONMENT
# ------------------------------------------------------------

envs = gym.vector.SyncVectorEnv(
    [
        make_env(
            args.env_id,
            args.seed + i,
            i,
            args.capture_video,
            run_name
        )
        for i in range(args.num_envs)
    ]
)

assert isinstance(
    envs.single_action_space,
    gym.spaces.Discrete
), "Only discrete action space is supported"


# ------------------------------------------------------------
# 9. AGENT + OPTIMIZER
# ------------------------------------------------------------

agent = Agent(envs).to(device)

optimizer = optim.Adam(
    agent.parameters(),
    lr=args.learning_rate,
    eps=1e-5
)

print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in agent.parameters()
    )
)


# ------------------------------------------------------------
# 10. ROLLOUT STORAGE
# ------------------------------------------------------------

obs = torch.zeros(
    (
        args.num_steps,
        args.num_envs
    )
    + envs.single_observation_space.shape
).to(device)

actions = torch.zeros(
    (
        args.num_steps,
        args.num_envs
    )
    + envs.single_action_space.shape
).to(device)

logprobs = torch.zeros(
    args.num_steps,
    args.num_envs
).to(device)

rewards = torch.zeros(
    args.num_steps,
    args.num_envs
).to(device)

dones = torch.zeros(
    args.num_steps,
    args.num_envs
).to(device)

values = torch.zeros(
    args.num_steps,
    args.num_envs
).to(device)


# ------------------------------------------------------------
# 11. TENSORBOARD
# ------------------------------------------------------------

writer = SummaryWriter(
    f"runs/{run_name}"
)

writer.add_text(
    "hyperparameters",
    "|param|value|\n|-|-|\n%s"
    %
    (
        "\n".join(
            [
                f"|{key}|{value}|"
                for key, value
                in vars(args).items()
            ]
        )
    ),
)


# ------------------------------------------------------------
# 12. START ENVIRONMENT
# ------------------------------------------------------------

global_step = 0
start_time = time.time()

next_obs = torch.Tensor(
    envs.reset()
).to(device)

next_done = torch.zeros(
    args.num_envs
).to(device)

num_updates = (
    args.total_timesteps
    // args.batch_size
)

print(
    "Number of PPO updates:",
    num_updates
)

print("=" * 60)


# ------------------------------------------------------------
# 13. PPO TRAINING LOOP
# ------------------------------------------------------------

for update in range(
    1,
    num_updates + 1
):

    # Anneal learning rate
    if args.anneal_lr:

        frac = (
            1.0
            - (update - 1.0)
            / num_updates
        )

        lrnow = (
            frac
            * args.learning_rate
        )

        optimizer.param_groups[0][
            "lr"
        ] = lrnow


    # --------------------------------------------------------
    # COLLECT ROLLOUT
    # --------------------------------------------------------

    for step in range(
        args.num_steps
    ):

        global_step += (
            1 * args.num_envs
        )

        obs[step] = next_obs
        dones[step] = next_done


        with torch.no_grad():

            (
                action,
                logprob,
                _,
                value
            ) = agent.get_action_and_value(
                next_obs
            )

            values[step] = (
                value.flatten()
            )


        actions[step] = action
        logprobs[step] = logprob


        next_obs, reward, done, info = (
            envs.step(
                action.cpu().numpy()
            )
        )


        rewards[step] = (
            torch.tensor(
                reward
            )
            .to(device)
            .view(-1)
        )


        next_obs = torch.Tensor(
            next_obs
        ).to(device)

        next_done = torch.Tensor(
            done
        ).to(device)


        # Episode statistics
        for item in info:

            if "episode" in item.keys():

                print(
                    f"global_step={global_step}, "
                    f"episodic_return="
                    f"{item['episode']['r']}"
                )

                writer.add_scalar(
                    "charts/episodic_return",
                    item["episode"]["r"],
                    global_step
                )

                writer.add_scalar(
                    "charts/episodic_length",
                    item["episode"]["l"],
                    global_step
                )

                break


    # --------------------------------------------------------
    # GAE
    # --------------------------------------------------------

    with torch.no_grad():

        next_value = (
            agent
            .get_value(next_obs)
            .reshape(1, -1)
        )

        if args.gae:

            advantages = torch.zeros_like(
                rewards
            ).to(device)

            lastgaelam = 0

            for t in reversed(
                range(args.num_steps)
            ):

                if t == args.num_steps - 1:

                    nextnonterminal = (
                        1.0 - next_done
                    )

                    nextvalues = next_value

                else:

                    nextnonterminal = (
                        1.0 - dones[t + 1]
                    )

                    nextvalues = (
                        values[t + 1]
                    )


                delta = (
                    rewards[t]
                    + args.gamma
                    * nextvalues
                    * nextnonterminal
                    - values[t]
                )


                advantages[t] = (
                    lastgaelam
                ) = (
                    delta
                    + args.gamma
                    * args.gae_lambda
                    * nextnonterminal
                    * lastgaelam
                )


            returns = (
                advantages
                + values
            )

        else:

            returns = torch.zeros_like(
                rewards
            ).to(device)

            for t in reversed(
                range(args.num_steps)
            ):

                if t == args.num_steps - 1:

                    nextnonterminal = (
                        1.0 - next_done
                    )

                    next_return = next_value

                else:

                    nextnonterminal = (
                        1.0 - dones[t + 1]
                    )

                    next_return = (
                        returns[t + 1]
                    )


                returns[t] = (
                    rewards[t]
                    + args.gamma
                    * nextnonterminal
                    * next_return
                )


            advantages = (
                returns - values
            )


    # --------------------------------------------------------
    # FLATTEN BATCH
    # --------------------------------------------------------

    b_obs = obs.reshape(
        (-1,)
        + envs.single_observation_space.shape
    )

    b_logprobs = (
        logprobs.reshape(-1)
    )

    b_actions = actions.reshape(
        (-1,)
        + envs.single_action_space.shape
    )

    b_advantages = (
        advantages.reshape(-1)
    )

    b_returns = (
        returns.reshape(-1)
    )

    b_values = (
        values.reshape(-1)
    )


    # --------------------------------------------------------
    # PPO UPDATE
    # --------------------------------------------------------

    b_inds = np.arange(
        args.batch_size
    )

    clipfracs = []


    for epoch in range(
        args.update_epochs
    ):

        np.random.shuffle(
            b_inds
        )


        for start in range(
            0,
            args.batch_size,
            args.minibatch_size
        ):

            end = (
                start
                + args.minibatch_size
            )

            mb_inds = b_inds[
                start:end
            ]


            (
                _,
                newlogprob,
                entropy,
                newvalue
            ) = agent.get_action_and_value(
                b_obs[mb_inds],
                b_actions.long()[mb_inds]
            )


            logratio = (
                newlogprob
                - b_logprobs[mb_inds]
            )

            ratio = logratio.exp()


            with torch.no_grad():

                old_approx_kl = (
                    -logratio
                ).mean()

                approx_kl = (
                    (ratio - 1)
                    - logratio
                ).mean()

                clipfracs.append(
                    (
                        (ratio - 1.0)
                        .abs()
                        > args.clip_coef
                    )
                    .float()
                    .mean()
                    .item()
                )


            mb_advantages = (
                b_advantages[mb_inds]
            )


            if args.norm_adv:

                mb_advantages = (
                    mb_advantages
                    - mb_advantages.mean()
                ) / (
                    mb_advantages.std()
                    + 1e-8
                )


            # Policy loss
            pg_loss1 = (
                -mb_advantages
                * ratio
            )

            pg_loss2 = (
                -mb_advantages
                * torch.clamp(
                    ratio,
                    1 - args.clip_coef,
                    1 + args.clip_coef
                )
            )

            pg_loss = torch.max(
                pg_loss1,
                pg_loss2
            ).mean()


            # Value loss
            newvalue = (
                newvalue.view(-1)
            )


            if args.clip_vloss:

                v_loss_unclipped = (
                    newvalue
                    - b_returns[mb_inds]
                ) ** 2


                v_clipped = (
                    b_values[mb_inds]
                    + torch.clamp(
                        newvalue
                        - b_values[mb_inds],
                        -args.clip_coef,
                        args.clip_coef
                    )
                )


                v_loss_clipped = (
                    v_clipped
                    - b_returns[mb_inds]
                ) ** 2


                v_loss_max = torch.max(
                    v_loss_unclipped,
                    v_loss_clipped
                )


                v_loss = (
                    0.5
                    * v_loss_max.mean()
                )

            else:

                v_loss = (
                    0.5
                    * (
                        newvalue
                        - b_returns[mb_inds]
                    ) ** 2
                ).mean()


            # Entropy
            entropy_loss = (
                entropy.mean()
            )


            # Total PPO loss
            loss = (
                pg_loss
                - args.ent_coef
                * entropy_loss
                + v_loss
                * args.vf_coef
            )


            optimizer.zero_grad()

            loss.backward()

            nn.utils.clip_grad_norm_(
                agent.parameters(),
                args.max_grad_norm
            )

            optimizer.step()


        if (
            args.target_kl is not None
            and approx_kl
            > args.target_kl
        ):
            break


    # --------------------------------------------------------
    # LOGGING
    # --------------------------------------------------------

    y_pred = (
        b_values
        .cpu()
        .numpy()
    )

    y_true = (
        b_returns
        .cpu()
        .numpy()
    )

    var_y = np.var(y_true)

    explained_var = (
        np.nan
        if var_y == 0
        else
        1
        - np.var(y_true - y_pred)
        / var_y
    )


    writer.add_scalar(
        "charts/learning_rate",
        optimizer.param_groups[0]["lr"],
        global_step
    )

    writer.add_scalar(
        "losses/value_loss",
        v_loss.item(),
        global_step
    )

    writer.add_scalar(
        "losses/policy_loss",
        pg_loss.item(),
        global_step
    )

    writer.add_scalar(
        "losses/entropy",
        entropy_loss.item(),
        global_step
    )

    writer.add_scalar(
        "losses/old_approx_kl",
        old_approx_kl.item(),
        global_step
    )

    writer.add_scalar(
        "losses/approx_kl",
        approx_kl.item(),
        global_step
    )

    writer.add_scalar(
        "losses/clipfrac",
        np.mean(clipfracs),
        global_step
    )

    writer.add_scalar(
        "losses/explained_variance",
        explained_var,
        global_step
    )


    sps = int(
        global_step
        / (time.time() - start_time)
    )

    writer.add_scalar(
        "charts/SPS",
        sps,
        global_step
    )


    print(
        f"Update {update}/{num_updates} | "
        f"Steps {global_step} | "
        f"SPS {sps}"
    )


# ------------------------------------------------------------
# 14. FINISH TRAINING
# ------------------------------------------------------------

envs.close()
writer.close()

# Create evaluation environment.
# Keep this alive because the later package_to_hub section
# uses eval_env.
eval_env = gym.make(
    args.env_id
)

print("=" * 60)
print("PPO TRAINING COMPLETE")
print("Environment:", args.env_id)
print("Model:", type(agent).__name__)
print("HF repo:", args.repo_id)
print("=" * 60)

## Add the Hugging Face Integration 🤗
- In order to push our model to the Hub, we need to define a function `package_to_hub`

- Add dependencies we need to push our model to the Hub

In [ ]:
from huggingface_hub import HfApi, upload_folder
from huggingface_hub.repocard import metadata_eval_result, metadata_save

from pathlib import Path
import datetime
import tempfile
import json
import shutil
import imageio

from wasabi import Printer
msg = Printer()

- Add new argument in `parse_args()` function to define the repo-id where we want to push the model.

In [ ]:
repo_id = "Tripura8928/ppo-LunarLander-v2"

print("Hugging Face repo:", repo_id)

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

- Next, we add the methods needed to push the model to the Hub

- These methods will:
  - `_evalutate_agent()`: evaluate the agent.
  - `_generate_model_card()`: generate the model card of your agent.
  - `_record_video()`: record a video of your agent.

In [ ]:
from pathlib import Path
import datetime
import tempfile
import json
import shutil
import imageio

print("Dependencies loaded successfully.")

In [ ]:
def package_to_hub(repo_id,
                model,
                hyperparameters,
                eval_env,
                video_fps=30,
                commit_message="Push agent to the Hub",
                token= None,
                logs=None
                ):
  """
  Evaluate, Generate a video and Upload a model to Hugging Face Hub.
  This method does the complete pipeline:
  - It evaluates the model
  - It generates the model card
  - It generates a replay video of the agent
  - It pushes everything to the hub
  :param repo_id: id of the model repository from the Hugging Face Hub
  :param model: trained model
  :param eval_env: environment used to evaluate the agent
  :param fps: number of fps for rendering the video
  :param commit_message: commit message
  :param logs: directory on local machine of tensorboard logs you'd like to upload
  """
  msg.info(
        "This function will save, evaluate, generate a video of your agent, "
        "create a model card and push everything to the hub. "
        "It might take up to 1min. \n "
        "This is a work in progress: if you encounter a bug, please open an issue."
    )
  # Step 1: Clone or create the repo
  repo_url = HfApi().create_repo(
        repo_id=repo_id,
        token=token,
        private=False,
        exist_ok=True,
    )

  with tempfile.TemporaryDirectory() as tmpdirname:
    tmpdirname = Path(tmpdirname)

    # Step 2: Save the model
    torch.save(model.state_dict(), tmpdirname / "model.pt")

    # Step 3: Evaluate the model and build JSON
    mean_reward, std_reward = _evaluate_agent(eval_env,
                                           10,
                                           model)

    # First get datetime
    eval_datetime = datetime.datetime.now()
    eval_form_datetime = eval_datetime.isoformat()

    evaluate_data = {
        "env_id": hyperparameters.env_id,
        "mean_reward": mean_reward,
        "std_reward": std_reward,
        "n_evaluation_episodes": 10,
        "eval_datetime": eval_form_datetime,
    }

    # Write a JSON file
    with open(tmpdirname / "results.json", "w") as outfile:
      json.dump(evaluate_data, outfile)

    # Step 4: Generate a video
    video_path =  tmpdirname / "replay.mp4"
    record_video(eval_env, model, video_path, video_fps)

    # Step 5: Generate the model card
    generated_model_card, metadata = _generate_model_card("PPO", hyperparameters.env_id, mean_reward, std_reward, hyperparameters)
    _save_model_card(tmpdirname, generated_model_card, metadata)

    # Step 6: Add logs if needed
    if logs:
      _add_logdir(tmpdirname, Path(logs))

    msg.info(f"Pushing repo {repo_id} to the Hugging Face Hub")

    repo_url = upload_folder(
            repo_id=repo_id,
            folder_path=tmpdirname,
            path_in_repo="",
            commit_message=commit_message,
            token=token,
        )

    msg.info(f"Your model is pushed to the Hub. You can view your model here: {repo_url}")
  return repo_url


def _evaluate_agent(env, n_eval_episodes, policy):
  """
  Evaluate the agent for ``n_eval_episodes`` episodes and returns average reward and std of reward.
  :param env: The evaluation environment
  :param n_eval_episodes: Number of episode to evaluate the agent
  :param policy: The agent
  """
  episode_rewards = []
  for episode in range(n_eval_episodes):
    state = env.reset()
    step = 0
    done = False
    total_rewards_ep = 0

    while done is False:
      state = torch.Tensor(state).to(device)
      action, _, _, _ = policy.get_action_and_value(state)
      new_state, reward, done, info = env.step(action.cpu().numpy())
      total_rewards_ep += reward
      if done:
        break
      state = new_state
    episode_rewards.append(total_rewards_ep)
  mean_reward = np.mean(episode_rewards)
  std_reward = np.std(episode_rewards)

  return mean_reward, std_reward


def record_video(env, policy, out_directory, fps=30):
  images = []
  done = False
  state = env.reset()
  img = env.render(mode='rgb_array')
  images.append(img)
  while not done:
    state = torch.Tensor(state).to(device)
    # Take the action (index) that have the maximum expected future reward given that state
    action, _, _, _  = policy.get_action_and_value(state)
    state, reward, done, info = env.step(action.cpu().numpy()) # We directly put next_state = state for recording logic
    img = env.render(mode='rgb_array')
    images.append(img)
  imageio.mimsave(out_directory, [np.array(img) for i, img in enumerate(images)], fps=fps)


def _generate_model_card(model_name, env_id, mean_reward, std_reward, hyperparameters):
  """
  Generate the model card for the Hub
  :param model_name: name of the model
  :env_id: name of the environment
  :mean_reward: mean reward of the agent
  :std_reward: standard deviation of the mean reward of the agent
  :hyperparameters: training arguments
  """
  # Step 1: Select the tags
  metadata = generate_metadata(model_name, env_id, mean_reward, std_reward)

  # Transform the hyperparams namespace to string
  converted_dict = vars(hyperparameters)
  converted_str = str(converted_dict)
  converted_str = converted_str.split(", ")
  converted_str = '\n'.join(converted_str)

  # Step 2: Generate the model card
  model_card = f"""
  # PPO Agent Playing {env_id}

  This is a trained model of a PPO agent playing {env_id}.

  # Hyperparameters
  ```python
  {converted_str}
  ```
  """
  return model_card, metadata


def generate_metadata(model_name, env_id, mean_reward, std_reward):
  """
  Define the tags for the model card
  :param model_name: name of the model
  :param env_id: name of the environment
  :mean_reward: mean reward of the agent
  :std_reward: standard deviation of the mean reward of the agent
  """
  metadata = {}
  metadata["tags"] = [
        env_id,
        "ppo",
        "deep-reinforcement-learning",
        "reinforcement-learning",
        "custom-implementation",
        "deep-rl-course"
  ]

  # Add metrics
  eval = metadata_eval_result(
      model_pretty_name=model_name,
      task_pretty_name="reinforcement-learning",
      task_id="reinforcement-learning",
      metrics_pretty_name="mean_reward",
      metrics_id="mean_reward",
      metrics_value=f"{mean_reward:.2f} +/- {std_reward:.2f}",
      dataset_pretty_name=env_id,
      dataset_id=env_id,
  )

  # Merges both dictionaries
  metadata = {**metadata, **eval}

  return metadata


def _save_model_card(local_path, generated_model_card, metadata):
    """Saves a model card for the repository.
    :param local_path: repository directory
    :param generated_model_card: model card generated by _generate_model_card()
    :param metadata: metadata
    """
    readme_path = local_path / "README.md"
    readme = ""
    if readme_path.exists():
        with readme_path.open("r", encoding="utf8") as f:
            readme = f.read()
    else:
        readme = generated_model_card

    with readme_path.open("w", encoding="utf-8") as f:
        f.write(readme)

    # Save our metrics to Readme metadata
    metadata_save(readme_path, metadata)


def _add_logdir(local_path: Path, logdir: Path):
  """Adds a logdir to the repository.
  :param local_path: repository directory
  :param logdir: logdir directory
  """
  if logdir.exists() and logdir.is_dir():
    # Add the logdir to the repository under new dir called logs
    repo_logdir = local_path / "logs"

    # Delete current logs if they exist
    if repo_logdir.exists():
      shutil.rmtree(repo_logdir)

    # Copy logdir into repo logdir
    shutil.copytree(logdir, repo_logdir)

- Finally, we call this function at the end of the PPO training

In [ ]:
print("gym:", "gym" in globals())
print("agent:", "agent" in globals())
print("args:", "args" in globals())

In [ ]:
!pip install -q setuptools==70.3.0 Box2D==2.3.10

import gym
import torch

print("Gym:", gym.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ============================================================
# PPO IMPLEMENTATION - UNIT 8
# Based on the official Hugging Face / CleanRL implementation
# ============================================================

import argparse
import os
import random
import time

import gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions.categorical import Categorical
from torch.utils.tensorboard import SummaryWriter


# ------------------------------------------------------------
# 1. ARGUMENTS
# ------------------------------------------------------------

def strtobool(value):
    value = str(value).lower()

    if value in ("y", "yes", "t", "true", "1"):
        return True

    if value in ("n", "no", "f", "false", "0"):
        return False

    raise ValueError(
        f"Invalid boolean value: {value}"
    )


def parse_args():

    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--exp-name",
        type=str,
        default="ppo-LunarLander-v2",
    )

    parser.add_argument(
        "--seed",
        type=int,
        default=1,
    )

    parser.add_argument(
        "--torch-deterministic",
        type=lambda x: strtobool(x),
        default=True,
        nargs="?",
        const=True,
    )

    parser.add_argument(
        "--cuda",
        type=lambda x: strtobool(x),
        default=True,
        nargs="?",
        const=True,
    )

    parser.add_argument(
        "--track",
        type=lambda x: strtobool(x),
        default=False,
        nargs="?",
        const=True,
    )

    parser.add_argument(
        "--wandb-project-name",
        type=str,
        default="cleanRL",
    )

    parser.add_argument(
        "--wandb-entity",
        type=str,
        default=None,
    )

    parser.add_argument(
        "--capture-video",
        type=lambda x: strtobool(x),
        default=False,
        nargs="?",
        const=True,
    )

    # PPO parameters

    parser.add_argument(
        "--env-id",
        type=str,
        default="LunarLander-v2",
    )

    parser.add_argument(
        "--total-timesteps",
        type=int,
        default=50000,
    )

    parser.add_argument(
        "--learning-rate",
        type=float,
        default=2.5e-4,
    )

    parser.add_argument(
        "--num-envs",
        type=int,
        default=4,
    )

    parser.add_argument(
        "--num-steps",
        type=int,
        default=128,
    )

    parser.add_argument(
        "--anneal-lr",
        type=lambda x: strtobool(x),
        default=True,
        nargs="?",
        const=True,
    )

    parser.add_argument(
        "--gae",
        type=lambda x: strtobool(x),
        default=True,
        nargs="?",
        const=True,
    )

    parser.add_argument(
        "--gamma",
        type=float,
        default=0.99,
    )

    parser.add_argument(
        "--gae-lambda",
        type=float,
        default=0.95,
    )

    parser.add_argument(
        "--num-minibatches",
        type=int,
        default=4,
    )

    parser.add_argument(
        "--update-epochs",
        type=int,
        default=4,
    )

    parser.add_argument(
        "--norm-adv",
        type=lambda x: strtobool(x),
        default=True,
        nargs="?",
        const=True,
    )

    parser.add_argument(
        "--clip-coef",
        type=float,
        default=0.2,
    )

    parser.add_argument(
        "--clip-vloss",
        type=lambda x: strtobool(x),
        default=True,
        nargs="?",
        const=True,
    )

    parser.add_argument(
        "--ent-coef",
        type=float,
        default=0.01,
    )

    parser.add_argument(
        "--vf-coef",
        type=float,
        default=0.5,
    )

    parser.add_argument(
        "--max-grad-norm",
        type=float,
        default=0.5,
    )

    parser.add_argument(
        "--target-kl",
        type=float,
        default=None,
    )

    # Hugging Face

    parser.add_argument(
        "--repo-id",
        type=str,
        default="Tripura8928/ppo-LunarLander-v2",
    )

    # Prevent Colab arguments from interfering
    args = parser.parse_args(args=[])

    args.batch_size = (
        args.num_envs * args.num_steps
    )

    args.minibatch_size = (
        args.batch_size
        // args.num_minibatches
    )

    return args


# ------------------------------------------------------------
# 2. ENVIRONMENT
# ------------------------------------------------------------

def make_env(
    env_id,
    seed,
    idx,
    capture_video,
    run_name
):

    def thunk():

        env = gym.make(env_id)

        env = gym.wrappers.RecordEpisodeStatistics(
            env
        )

        if capture_video and idx == 0:

            env = gym.wrappers.RecordVideo(
                env,
                f"videos/{run_name}"
            )

        env.seed(seed)

        env.action_space.seed(seed)

        env.observation_space.seed(seed)

        return env

    return thunk


# ------------------------------------------------------------
# 3. NETWORK INITIALIZATION
# ------------------------------------------------------------

def layer_init(
    layer,
    std=np.sqrt(2),
    bias_const=0.0
):

    torch.nn.init.orthogonal_(
        layer.weight,
        std
    )

    torch.nn.init.constant_(
        layer.bias,
        bias_const
    )

    return layer


# ------------------------------------------------------------
# 4. PPO AGENT
# ------------------------------------------------------------

class Agent(nn.Module):

    def __init__(self, envs):

        super().__init__()

        self.critic = nn.Sequential(

            layer_init(
                nn.Linear(
                    np.array(
                        envs.single_observation_space.shape
                    ).prod(),
                    64
                )
            ),

            nn.Tanh(),

            layer_init(
                nn.Linear(64, 64)
            ),

            nn.Tanh(),

            layer_init(
                nn.Linear(64, 1),
                std=1.0
            ),
        )

        self.actor = nn.Sequential(

            layer_init(
                nn.Linear(
                    np.array(
                        envs.single_observation_space.shape
                    ).prod(),
                    64
                )
            ),

            nn.Tanh(),

            layer_init(
                nn.Linear(64, 64)
            ),

            nn.Tanh(),

            layer_init(
                nn.Linear(
                    64,
                    envs.single_action_space.n
                ),
                std=0.01
            ),
        )

    def get_value(self, x):

        return self.critic(x)

    def get_action_and_value(
        self,
        x,
        action=None
    ):

        logits = self.actor(x)

        probs = Categorical(
            logits=logits
        )

        if action is None:

            action = probs.sample()

        return (
            action,
            probs.log_prob(action),
            probs.entropy(),
            self.critic(x),
        )


# ------------------------------------------------------------
# 5. START PPO
# ------------------------------------------------------------

args = parse_args()

run_name = (
    f"{args.env_id}__"
    f"{args.exp_name}__"
    f"{args.seed}__"
    f"{int(time.time())}"
)

print("=" * 60)
print("PPO TRAINING")
print("=" * 60)
print("Environment:", args.env_id)
print("Total timesteps:", args.total_timesteps)
print("Parallel environments:", args.num_envs)
print("Steps per rollout:", args.num_steps)
print("Batch size:", args.batch_size)
print("Minibatch size:", args.minibatch_size)
print("Repository:", args.repo_id)


# ------------------------------------------------------------
# 6. SEEDING
# ------------------------------------------------------------

random.seed(args.seed)
np.random.seed(args.seed)
torch.manual_seed(args.seed)

torch.backends.cudnn.deterministic = (
    args.torch_deterministic
)


# ------------------------------------------------------------
# 7. DEVICE
# ------------------------------------------------------------

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    and args.cuda
    else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


# ------------------------------------------------------------
# 8. VECTOR ENVIRONMENT
# ------------------------------------------------------------

envs = gym.vector.SyncVectorEnv(
    [
        make_env(
            args.env_id,
            args.seed + i,
            i,
            args.capture_video,
            run_name
        )
        for i in range(args.num_envs)
    ]
)

assert isinstance(
    envs.single_action_space,
    gym.spaces.Discrete
), "Only discrete action space is supported"


# ------------------------------------------------------------
# 9. AGENT + OPTIMIZER
# ------------------------------------------------------------

agent = Agent(envs).to(device)

optimizer = optim.Adam(
    agent.parameters(),
    lr=args.learning_rate,
    eps=1e-5
)

print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in agent.parameters()
    )
)


# ------------------------------------------------------------
# 10. ROLLOUT STORAGE
# ------------------------------------------------------------

obs = torch.zeros(
    (
        args.num_steps,
        args.num_envs
    )
    + envs.single_observation_space.shape
).to(device)

actions = torch.zeros(
    (
        args.num_steps,
        args.num_envs
    )
    + envs.single_action_space.shape
).to(device)

logprobs = torch.zeros(
    args.num_steps,
    args.num_envs
).to(device)

rewards = torch.zeros(
    args.num_steps,
    args.num_envs
).to(device)

dones = torch.zeros(
    args.num_steps,
    args.num_envs
).to(device)

values = torch.zeros(
    args.num_steps,
    args.num_envs
).to(device)


# ------------------------------------------------------------
# 11. TENSORBOARD
# ------------------------------------------------------------

writer = SummaryWriter(
    f"runs/{run_name}"
)

writer.add_text(
    "hyperparameters",
    "|param|value|\n|-|-|\n%s"
    %
    (
        "\n".join(
            [
                f"|{key}|{value}|"
                for key, value
                in vars(args).items()
            ]
        )
    ),
)


# ------------------------------------------------------------
# 12. START ENVIRONMENT
# ------------------------------------------------------------

global_step = 0

start_time = time.time()

next_obs = torch.Tensor(
    envs.reset()
).to(device)

next_done = torch.zeros(
    args.num_envs
).to(device)

num_updates = (
    args.total_timesteps
    // args.batch_size
)

print(
    "Number of PPO updates:",
    num_updates
)

print("=" * 60)


# ------------------------------------------------------------
# 13. PPO TRAINING LOOP
# ------------------------------------------------------------

for update in range(
    1,
    num_updates + 1
):

    # Anneal learning rate

    if args.anneal_lr:

        frac = (
            1.0
            - (update - 1.0)
            / num_updates
        )

        lrnow = (
            frac
            * args.learning_rate
        )

        optimizer.param_groups[0][
            "lr"
        ] = lrnow


    # --------------------------------------------------------
    # COLLECT ROLLOUT
    # --------------------------------------------------------

    for step in range(
        args.num_steps
    ):

        global_step += (
            1 * args.num_envs
        )

        obs[step] = next_obs

        dones[step] = next_done


        with torch.no_grad():

            (
                action,
                logprob,
                _,
                value
            ) = agent.get_action_and_value(
                next_obs
            )

            values[step] = (
                value.flatten()
            )


        actions[step] = action

        logprobs[step] = logprob


        next_obs, reward, done, info = (
            envs.step(
                action.cpu().numpy()
            )
        )


        rewards[step] = (
            torch.tensor(
                reward
            )
            .to(device)
            .view(-1)
        )


        next_obs = torch.Tensor(
            next_obs
        ).to(device)

        next_done = torch.Tensor(
            done
        ).to(device)


        # Episode statistics

        for item in info:

            if "episode" in item.keys():

                print(
                    f"global_step={global_step}, "
                    f"episodic_return="
                    f"{item['episode']['r']}"
                )

                writer.add_scalar(
                    "charts/episodic_return",
                    item["episode"]["r"],
                    global_step
                )

                writer.add_scalar(
                    "charts/episodic_length",
                    item["episode"]["l"],
                    global_step
                )

                break


    # --------------------------------------------------------
    # GAE
    # --------------------------------------------------------

    with torch.no_grad():

        next_value = (
            agent
            .get_value(next_obs)
            .reshape(1, -1)
        )

        if args.gae:

            advantages = torch.zeros_like(
                rewards
            ).to(device)

            lastgaelam = 0

            for t in reversed(
                range(args.num_steps)
            ):

                if t == args.num_steps - 1:

                    nextnonterminal = (
                        1.0 - next_done
                    )

                    nextvalues = next_value

                else:

                    nextnonterminal = (
                        1.0 - dones[t + 1]
                    )

                    nextvalues = (
                        values[t + 1]
                    )


                delta = (
                    rewards[t]
                    + args.gamma
                    * nextvalues
                    * nextnonterminal
                    - values[t]
                )


                advantages[t] = (
                    lastgaelam
                ) = (
                    delta
                    + args.gamma
                    * args.gae_lambda
                    * nextnonterminal
                    * lastgaelam
                )


            returns = (
                advantages
                + values
            )

        else:

            returns = torch.zeros_like(
                rewards
            ).to(device)

            for t in reversed(
                range(args.num_steps)
            ):

                if t == args.num_steps - 1:

                    nextnonterminal = (
                        1.0 - next_done
                    )

                    next_return = next_value

                else:

                    nextnonterminal = (
                        1.0 - dones[t + 1]
                    )

                    next_return = (
                        returns[t + 1]
                    )


                returns[t] = (
                    rewards[t]
                    + args.gamma
                    * nextnonterminal
                    * next_return
                )


            advantages = (
                returns - values
            )


    # --------------------------------------------------------
    # FLATTEN BATCH
    # --------------------------------------------------------

    b_obs = obs.reshape(
        (-1,)
        + envs.single_observation_space.shape
    )

    b_logprobs = (
        logprobs.reshape(-1)
    )

    b_actions = actions.reshape(
        (-1,)
        + envs.single_action_space.shape
    )

    b_advantages = (
        advantages.reshape(-1)
    )

    b_returns = (
        returns.reshape(-1)
    )

    b_values = (
        values.reshape(-1)
    )


    # --------------------------------------------------------
    # PPO UPDATE
    # --------------------------------------------------------

    b_inds = np.arange(
        args.batch_size
    )

    clipfracs = []


    for epoch in range(
        args.update_epochs
    ):

        np.random.shuffle(
            b_inds
        )


        for start in range(
            0,
            args.batch_size,
            args.minibatch_size
        ):

            end = (
                start
                + args.minibatch_size
            )

            mb_inds = b_inds[
                start:end
            ]


            (
                _,
                newlogprob,
                entropy,
                newvalue
            ) = agent.get_action_and_value(
                b_obs[mb_inds],
                b_actions.long()[mb_inds]
            )


            logratio = (
                newlogprob
                - b_logprobs[mb_inds]
            )

            ratio = logratio.exp()


            with torch.no_grad():

                old_approx_kl = (
                    -logratio
                ).mean()

                approx_kl = (
                    (ratio - 1)
                    - logratio
                ).mean()

                clipfracs.append(
                    (
                        (ratio - 1.0)
                        .abs()
                        > args.clip_coef
                    )
                    .float()
                    .mean()
                    .item()
                )


            mb_advantages = (
                b_advantages[mb_inds]
            )


            if args.norm_adv:

                mb_advantages = (
                    mb_advantages
                    - mb_advantages.mean()
                ) / (
                    mb_advantages.std()
                    + 1e-8
                )


            # Policy loss

            pg_loss1 = (
                -mb_advantages
                * ratio
            )

            pg_loss2 = (
                -mb_advantages
                * torch.clamp(
                    ratio,
                    1 - args.clip_coef,
                    1 + args.clip_coef
                )
            )

            pg_loss = torch.max(
                pg_loss1,
                pg_loss2
            ).mean()


            # Value loss

            newvalue = (
                newvalue.view(-1)
            )


            if args.clip_vloss:

                v_loss_unclipped = (
                    newvalue
                    - b_returns[mb_inds]
                ) ** 2


                v_clipped = (
                    b_values[mb_inds]
                    + torch.clamp(
                        newvalue
                        - b_values[mb_inds],
                        -args.clip_coef,
                        args.clip_coef
                    )
                )


                v_loss_clipped = (
                    v_clipped
                    - b_returns[mb_inds]
                ) ** 2


                v_loss_max = torch.max(
                    v_loss_unclipped,
                    v_loss_clipped
                )


                v_loss = (
                    0.5
                    * v_loss_max.mean()
                )

            else:

                v_loss = (
                    0.5
                    * (
                        newvalue
                        - b_returns[mb_inds]
                    ) ** 2
                ).mean()


            # Entropy

            entropy_loss = (
                entropy.mean()
            )


            # Total PPO loss

            loss = (
                pg_loss
                - args.ent_coef
                * entropy_loss
                + v_loss
                * args.vf_coef
            )


            optimizer.zero_grad()

            loss.backward()

            nn.utils.clip_grad_norm_(
                agent.parameters(),
                args.max_grad_norm
            )

            optimizer.step()


        if (
            args.target_kl is not None
            and approx_kl
            > args.target_kl
        ):

            break


    # --------------------------------------------------------
    # LOGGING
    # --------------------------------------------------------

    y_pred = (
        b_values
        .cpu()
        .numpy()
    )

    y_true = (
        b_returns
        .cpu()
        .numpy()
    )

    var_y = np.var(y_true)

    explained_var = (
        np.nan
        if var_y == 0
        else
        1
        - np.var(y_true - y_pred)
        / var_y
    )


    writer.add_scalar(
        "charts/learning_rate",
        optimizer.param_groups[0]["lr"],
        global_step
    )

    writer.add_scalar(
        "losses/value_loss",
        v_loss.item(),
        global_step
    )

    writer.add_scalar(
        "losses/policy_loss",
        pg_loss.item(),
        global_step
    )

    writer.add_scalar(
        "losses/entropy",
        entropy_loss.item(),
        global_step
    )

    writer.add_scalar(
        "losses/old_approx_kl",
        old_approx_kl.item(),
        global_step
    )

    writer.add_scalar(
        "losses/approx_kl",
        approx_kl.item(),
        global_step
    )

    writer.add_scalar(
        "losses/clipfrac",
        np.mean(clipfracs),
        global_step
    )

    writer.add_scalar(
        "losses/explained_variance",
        explained_var,
        global_step
    )


    sps = int(
        global_step
        / (time.time() - start_time)
    )

    writer.add_scalar(
        "charts/SPS",
        sps,
        global_step
    )


    print(
        f"Update {update}/{num_updates} | "
        f"Steps {global_step} | "
        f"SPS {sps}"
    )


# ------------------------------------------------------------
# 14. SAVE MODEL IMMEDIATELY
# ------------------------------------------------------------

# IMPORTANT:
# Save before doing any Hugging Face packaging/evaluation.
# This protects the trained model if the Colab runtime crashes.

model_path = (
    "ppo-LunarLander-v2.pt"
)

torch.save(
    agent.state_dict(),
    model_path
)

# Save the training configuration too.

config_path = (
    "ppo-LunarLander-v2-config.json"
)

with open(
    config_path,
    "w"
) as f:

    import json

    json.dump(
        vars(args),
        f,
        indent=2
    )

print("=" * 60)
print("PPO TRAINING COMPLETE")
print("=" * 60)
print("Environment:", args.env_id)
print("Model:", type(agent).__name__)
print("HF repo:", args.repo_id)
print()
print("MODEL SAVED:")
print(model_path)
print()
print("CONFIG SAVED:")
print(config_path)
print("=" * 60)


# ------------------------------------------------------------
# 15. CLOSE TRAINING ENVIRONMENT
# ------------------------------------------------------------

envs.close()

writer.close()


# ------------------------------------------------------------
# 16. CREATE EVALUATION ENVIRONMENT
# ------------------------------------------------------------

eval_env = gym.make(
    args.env_id
)

print("Evaluation environment created.")
print("Ready for package_to_hub().")

In [ ]:
package_to_hub(
    repo_id="Tripura8928/ppo-LunarLander-v2",
    model=agent,
    hyperparameters=args,
    eval_env=eval_env,
    logs=f"runs/{run_name}",
)

In [ ]:
# Create the evaluation environment
eval_env = gym.make(args.env_id)

package_to_hub(repo_id = args.repo_id,
                model = agent, # The model we want to save
                hyperparameters = args,
                eval_env = gym.make(args.env_id),
                logs= f"runs/{run_name}",
                )

- Here's what look the ppo.py final file

To be able to share your model with the community there are three more steps to follow:

1️⃣ (If it's not already done) create an account to HF ➡ https://huggingface.co/join

2️⃣ Sign in and then, you need to store your authentication token from the Hugging Face website.
- Create a new token (https://huggingface.co/settings/tokens) **with write role**

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/create-token.jpg" alt="Create HF Token">

- Copy the token
- Run the cell below and paste the token

In [ ]:
# ============================================================
# UNIT 8 - PPO LunarLander-v2
# COMPLETE PACKAGING + HUGGING FACE HUB UPLOAD
# ============================================================

import os
import json
import shutil
import tempfile
import datetime
from pathlib import Path

import gym
import imageio
import numpy as np
import torch

from huggingface_hub import HfApi, upload_folder, notebook_login


# ------------------------------------------------------------
# 1. BASIC CHECKS
# ------------------------------------------------------------

print("Checking current Colab runtime...")

required_vars = ["agent", "args", "run_name"]

missing = [v for v in required_vars if v not in globals()]

if missing:
    raise RuntimeError(
        f"Missing variables: {missing}\n\n"
        "Your PPO training variables are not currently in memory. "
        "Please rerun the PPO training cell first, then run this cell."
    )

print("✓ PPO agent found")
print("✓ Training arguments found")
print(f"✓ Run name: {run_name}")


# ------------------------------------------------------------
# 2. HUGGING FACE LOGIN
# ------------------------------------------------------------

print("\nChecking Hugging Face authentication...")

try:
    api = HfApi()
    user = api.whoami()

    print(f"✓ Logged in as: {user['name']}")

except Exception:
    print("Please login to Hugging Face...")
    notebook_login()
    api = HfApi()

    user = api.whoami()
    print(f"✓ Logged in as: {user['name']}")


# ------------------------------------------------------------
# 3. CREATE EVALUATION ENVIRONMENT
# ------------------------------------------------------------

if "eval_env" not in globals() or eval_env is None:

    print("\nCreating evaluation environment...")

    eval_env = gym.make(args.env_id)

    print(f"✓ Evaluation environment: {args.env_id}")

else:

    print("\n✓ Evaluation environment already exists")


# ------------------------------------------------------------
# 4. EVALUATE AGENT
# ------------------------------------------------------------

def evaluate_agent(agent, env, episodes=10):

    print("\nEvaluating PPO agent...")

    rewards = []

    agent.eval()

    for episode in range(episodes):

        observation = env.reset()

        # Gym 0.22 returns only observation
        if isinstance(observation, tuple):
            observation = observation[0]

        done = False
        episode_reward = 0.0

        while not done:

            observation_tensor = torch.tensor(
                observation,
                dtype=torch.float32,
                device=next(agent.parameters()).device
            ).unsqueeze(0)

            with torch.no_grad():

                action, _, _, _ = agent.get_action_and_value(
                    observation_tensor
                )

            action = action.cpu().numpy()[0]

            step_result = env.step(action)

            if len(step_result) == 5:
                observation, reward, terminated, truncated, _ = step_result
                done = terminated or truncated
            else:
                observation, reward, done, _ = step_result

            episode_reward += reward

        rewards.append(episode_reward)

        print(
            f"Episode {episode + 1:2d}/{episodes}: "
            f"{episode_reward:.2f}"
        )

    agent.train()

    mean_reward = float(np.mean(rewards))
    std_reward = float(np.std(rewards))

    print("\nEvaluation complete!")
    print(f"Mean reward : {mean_reward:.2f}")
    print(f"Std reward  : {std_reward:.2f}")

    return mean_reward, std_reward


# ------------------------------------------------------------
# 5. RECORD VIDEO
# ------------------------------------------------------------

def record_video(agent, env_id, video_path, max_steps=3000):

    print("\nRecording evaluation video...")

    video_env = gym.make(env_id)

    frames = []

    observation = video_env.reset()

    if isinstance(observation, tuple):
        observation = observation[0]

    done = False
    step = 0

    while not done and step < max_steps:

        try:
            frame = video_env.render(mode="rgb_array")
        except TypeError:
            frame = video_env.render()

        if frame is not None:
            frames.append(frame)

        observation_tensor = torch.tensor(
            observation,
            dtype=torch.float32,
            device=next(agent.parameters()).device
        ).unsqueeze(0)

        with torch.no_grad():

            action, _, _, _ = agent.get_action_and_value(
                observation_tensor
            )

        action = action.cpu().numpy()[0]

        step_result = video_env.step(action)

        if len(step_result) == 5:
            observation, reward, terminated, truncated, _ = step_result
            done = terminated or truncated
        else:
            observation, reward, done, _ = step_result

        step += 1

    video_env.close()

    if len(frames) > 0:

        imageio.mimsave(
            video_path,
            frames,
            fps=30
        )

        print(f"✓ Video saved: {video_path}")

    else:

        print("⚠ No video frames were produced.")


# ------------------------------------------------------------
# 6. COMPLETE package_to_hub FUNCTION
# ------------------------------------------------------------

def package_to_hub(
    repo_id,
    model,
    hyperparameters,
    eval_env,
    logs=None
):

    print("\n" + "=" * 60)
    print("PACKAGING MODEL FOR HUGGING FACE HUB")
    print("=" * 60)

    # --------------------------------------------------------
    # Temporary package directory
    # --------------------------------------------------------

    package_dir = Path(tempfile.mkdtemp())

    print(f"\nTemporary package directory:")
    print(package_dir)

    # --------------------------------------------------------
    # Save PyTorch model
    # --------------------------------------------------------

    model_file = package_dir / "ppo-LunarLander-v2.pt"

    torch.save(
        model.state_dict(),
        model_file
    )

    print("✓ Model weights saved")

    # --------------------------------------------------------
    # Save hyperparameters
    # --------------------------------------------------------

    config_file = package_dir / "ppo-LunarLander-v2-config.json"

    config = {}

    try:
        config = vars(hyperparameters)
    except Exception:
        config = {
            "env_id": "LunarLander-v2",
            "total_timesteps": 50000
        }

    with open(config_file, "w") as f:
        json.dump(
            config,
            f,
            indent=2,
            default=str
        )

    print("✓ Configuration saved")

    # --------------------------------------------------------
    # Evaluation
    # --------------------------------------------------------

    mean_reward, std_reward = evaluate_agent(
        model,
        eval_env,
        episodes=10
    )

    # --------------------------------------------------------
    # Save evaluation results
    # --------------------------------------------------------

    evaluation_file = package_dir / "evaluation.json"

    evaluation_data = {
        "environment": "LunarLander-v2",
        "mean_reward": mean_reward,
        "std_reward": std_reward,
        "episodes": 10,
        "evaluation_date": datetime.datetime.now().isoformat()
    }

    with open(evaluation_file, "w") as f:
        json.dump(
            evaluation_data,
            f,
            indent=2
        )

    print("✓ Evaluation results saved")

    # --------------------------------------------------------
    # Video
    # --------------------------------------------------------

    video_file = package_dir / "replay.mp4"

    try:

        record_video(
            model,
            "LunarLander-v2",
            str(video_file)
        )

    except Exception as e:

        print(f"⚠ Video generation failed: {e}")

    # --------------------------------------------------------
    # README / MODEL CARD
    # --------------------------------------------------------

    readme_file = package_dir / "README.md"

    readme = f"""---
tags:
- deep-reinforcement-learning
- reinforcement-learning
- lunarlander
- pytorch
- stable-baselines3
- huggingface
- unit8
library_name: pytorch
---

# PPO LunarLander-v2

This repository contains a **Proximal Policy Optimization (PPO)** agent
trained from scratch using PyTorch on the `LunarLander-v2` environment.

## Environment

- Environment: LunarLander-v2
- Algorithm: PPO
- Framework: PyTorch
- Parallel environments: 4
- Rollout steps: 128
- Total training timesteps: {config.get("total_timesteps", 50000)}

## Evaluation

Mean reward: **{mean_reward:.2f}**

Standard deviation: **{std_reward:.2f}**

## Files

- `ppo-LunarLander-v2.pt` — trained PyTorch model
- `ppo-LunarLander-v2-config.json` — training configuration
- `evaluation.json` — evaluation results
- `replay.mp4` — evaluation gameplay video

## Hugging Face Deep Reinforcement Learning Course

This model was trained as part of **Unit 8: PPO with PyTorch**
of the Hugging Face Deep Reinforcement Learning Course.
"""

    with open(readme_file, "w") as f:
        f.write(readme)

    print("✓ README/model card created")

    # --------------------------------------------------------
    # Upload repository
    # --------------------------------------------------------

    print("\nCreating / updating Hugging Face repository...")

    api = HfApi()

    api.create_repo(
        repo_id=repo_id,
        repo_type="model",
        exist_ok=True
    )

    print(f"✓ Repository ready: {repo_id}")

    print("\nUploading files...")

    upload_folder(
        folder_path=str(package_dir),
        repo_id=repo_id,
        repo_type="model",
        commit_message="Upload PPO LunarLander-v2 trained model"
    )

    print("\n" + "=" * 60)
    print("🎉 UPLOAD COMPLETE!")
    print("=" * 60)

    print(f"\nHugging Face repository:")
    print(f"https://huggingface.co/{repo_id}")

    print("\nEvaluation:")
    print(f"Mean reward = {mean_reward:.2f}")
    print(f"Std reward  = {std_reward:.2f}")

    return {
        "repo_id": repo_id,
        "mean_reward": mean_reward,
        "std_reward": std_reward,
        "package_dir": str(package_dir)
    }


# ------------------------------------------------------------
# 7. RUN PACKAGE + UPLOAD
# ------------------------------------------------------------

result = package_to_hub(
    repo_id="Tripura8928/ppo-LunarLander-v2",
    model=agent,
    hyperparameters=args,
    eval_env=eval_env,
    logs=f"runs/{run_name}"
)

print("\n✅ Unit 8 PPO model packaging finished successfully.")

**COMPLETE UNIT 8 MODEL TRAINING FLOW**

In [ ]:
# ================================================================
# UNIT 8 - PPO FROM SCRATCH WITH PYTORCH
# LUNARLANDER-v2
# ONE CELL: TRAIN + EVALUATE + VIDEO + HUGGING FACE UPLOAD
# ================================================================

import os
import json
import time
import datetime
import shutil
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions.categorical import Categorical

import gym

from huggingface_hub import HfApi, upload_folder, notebook_login


# ================================================================
# 1. BASIC SETUP
# ================================================================

print("=" * 70)
print("UNIT 8 - PPO WITH PYTORCH")
print("=" * 70)

print(f"Gym version   : {gym.__version__}")
print(f"PyTorch       : {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"GPU           : {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("Using CPU")


# ================================================================
# 2. HUGGING FACE LOGIN
# ================================================================

print("\n" + "=" * 70)
print("HUGGING FACE LOGIN")
print("=" * 70)

try:
    api = HfApi()
    me = api.whoami()
    print(f"Already logged in as: {me['name']}")
except Exception:
    notebook_login()
    api = HfApi()
    me = api.whoami()
    print(f"Logged in as: {me['name']}")


# ================================================================
# 3. PPO CONFIGURATION
# ================================================================

ENV_ID = "LunarLander-v2"
TOTAL_TIMESTEPS = 50_000

NUM_ENVS = 4
NUM_STEPS = 128

LEARNING_RATE = 2.5e-4
GAMMA = 0.99
GAE_LAMBDA = 0.95

NUM_MINIBATCHES = 4
UPDATE_EPOCHS = 4

CLIP_COEF = 0.2
ENT_COEF = 0.01
VF_COEF = 0.5
MAX_GRAD_NORM = 0.5

SEED = 1

REPO_ID = "Tripura8928/ppo-LunarLander-v2"

run_name = (
    f"{ENV_ID}__ppo-LunarLander-v2__1__"
    f"{int(time.time())}"
)

print("\nTraining configuration:")
print(f"Environment      : {ENV_ID}")
print(f"Parallel envs    : {NUM_ENVS}")
print(f"Steps/rollout    : {NUM_STEPS}")
print(f"Total timesteps  : {TOTAL_TIMESTEPS}")
print(f"Learning rate    : {LEARNING_RATE}")
print(f"Gamma            : {GAMMA}")
print(f"GAE lambda       : {GAE_LAMBDA}")
print(f"HF repository    : {REPO_ID}")


# ================================================================
# 4. ENVIRONMENT
# ================================================================

def make_env(env_id, seed):

    def thunk():

        env = gym.make(env_id)

        env.seed(seed)
        env.action_space.seed(seed)
        env.observation_space.seed(seed)

        return env

    return thunk


envs = gym.vector.SyncVectorEnv(
    [
        make_env(
            ENV_ID,
            SEED + i
        )
        for i in range(NUM_ENVS)
    ]
)

print("\nEnvironment created.")
print("Observation space:", envs.single_observation_space)
print("Action space     :", envs.single_action_space)


# ================================================================
# 5. NETWORK INITIALIZATION
# ================================================================

def layer_init(
    layer,
    std=np.sqrt(2),
    bias_const=0.0
):

    torch.nn.init.orthogonal_(
        layer.weight,
        std
    )

    torch.nn.init.constant_(
        layer.bias,
        bias_const
    )

    return layer


# ================================================================
# 6. PPO AGENT
# ================================================================

class Agent(nn.Module):

    def __init__(self, envs):

        super().__init__()

        obs_dim = int(
            np.prod(
                envs.single_observation_space.shape
            )
        )

        action_dim = envs.single_action_space.n

        self.critic = nn.Sequential(

            layer_init(
                nn.Linear(
                    obs_dim,
                    64
                )
            ),

            nn.Tanh(),

            layer_init(
                nn.Linear(
                    64,
                    64
                )
            ),

            nn.Tanh(),

            layer_init(
                nn.Linear(
                    64,
                    1
                ),
                std=1.0
            )
        )

        self.actor = nn.Sequential(

            layer_init(
                nn.Linear(
                    obs_dim,
                    64
                )
            ),

            nn.Tanh(),

            layer_init(
                nn.Linear(
                    64,
                    64
                )
            ),

            nn.Tanh(),

            layer_init(
                nn.Linear(
                    64,
                    action_dim
                ),
                std=0.01
            )
        )

    def get_value(self, x):

        return self.critic(x)

    def get_action_and_value(
        self,
        x,
        action=None
    ):

        logits = self.actor(x)

        probs = Categorical(
            logits=logits
        )

        if action is None:
            action = probs.sample()

        return (
            action,
            probs.log_prob(action),
            probs.entropy(),
            self.critic(x)
        )


agent = Agent(envs).to(device)

optimizer = optim.Adam(
    agent.parameters(),
    lr=LEARNING_RATE,
    eps=1e-5
)

print("\nPPO agent created.")

print(
    "Trainable parameters:",
    sum(
        p.numel()
        for p in agent.parameters()
        if p.requires_grad
    )
)


# ================================================================
# 7. ROLLOUT STORAGE
# ================================================================

batch_size = NUM_ENVS * NUM_STEPS

minibatch_size = (
    batch_size // NUM_MINIBATCHES
)

obs = torch.zeros(
    (
        NUM_STEPS,
        NUM_ENVS
    )
    + envs.single_observation_space.shape
).to(device)

actions = torch.zeros(
    NUM_STEPS,
    NUM_ENVS
).to(device)

logprobs = torch.zeros(
    NUM_STEPS,
    NUM_ENVS
).to(device)

rewards = torch.zeros(
    NUM_STEPS,
    NUM_ENVS
).to(device)

dones = torch.zeros(
    NUM_STEPS,
    NUM_ENVS
).to(device)

values = torch.zeros(
    NUM_STEPS,
    NUM_ENVS
).to(device)


# ================================================================
# 8. RESET ENVIRONMENTS
# ================================================================

next_obs = envs.reset()

if isinstance(next_obs, tuple):
    next_obs = next_obs[0]

next_obs = torch.tensor(
    next_obs,
    dtype=torch.float32,
    device=device
)

next_done = torch.zeros(
    NUM_ENVS,
    device=device
)

global_step = 0

num_updates = (
    TOTAL_TIMESTEPS // batch_size
)

print("\n" + "=" * 70)
print("STARTING PPO TRAINING")
print("=" * 70)

start_time = time.time()


# ================================================================
# 9. PPO TRAINING
# ================================================================

for update in range(
    1,
    num_updates + 1
):

    # ------------------------------------------------------------
    # COLLECT ROLLOUT
    # ------------------------------------------------------------

    for step in range(NUM_STEPS):

        global_step += NUM_ENVS

        obs[step] = next_obs

        dones[step] = next_done

        with torch.no_grad():

            action, logprob, _, value = (
                agent.get_action_and_value(
                    next_obs
                )
            )

            values[step] = (
                value.flatten()
            )

        actions[step] = action

        logprobs[step] = logprob

        next_obs_np, reward, terminated, info = (
            envs.step(
                action.cpu().numpy()
            )
        )

        next_obs = torch.tensor(
            next_obs_np,
            dtype=torch.float32,
            device=device
        )

        rewards[step] = torch.tensor(
            reward,
            dtype=torch.float32,
            device=device
        )

        next_done = torch.tensor(
            terminated,
            dtype=torch.float32,
            device=device
        )


    # ------------------------------------------------------------
    # GENERALIZED ADVANTAGE ESTIMATION
    # ------------------------------------------------------------

    with torch.no_grad():

        next_value = agent.get_value(
            next_obs
        ).reshape(1, -1)

        advantages = torch.zeros_like(
            rewards
        )

        lastgaelam = torch.zeros(
            NUM_ENVS,
            device=device
        )

        for t in reversed(
            range(NUM_STEPS)
        ):

            if t == NUM_STEPS - 1:

                nextnonterminal = (
                    1.0 - next_done
                )

                nextvalues = next_value

            else:

                nextnonterminal = (
                    1.0 - dones[t + 1]
                )

                nextvalues = values[t + 1]

            delta = (
                rewards[t]
                + GAMMA
                * nextvalues
                * nextnonterminal
                - values[t]
            )

            lastgaelam = (
                delta
                + GAMMA
                * GAE_LAMBDA
                * nextnonterminal
                * lastgaelam
            )

            advantages[t] = lastgaelam

        returns = (
            advantages
            + values
        )


    # ------------------------------------------------------------
    # FLATTEN BATCH
    # ------------------------------------------------------------

    b_obs = obs.reshape(
        (-1,)
        + envs.single_observation_space.shape
    )

    b_logprobs = logprobs.reshape(-1)

    b_actions = actions.reshape(-1)

    b_advantages = advantages.reshape(-1)

    b_returns = returns.reshape(-1)

    b_values = values.reshape(-1)


    # ------------------------------------------------------------
    # PPO UPDATE
    # ------------------------------------------------------------

    b_inds = np.arange(
        batch_size
    )

    for epoch in range(
        UPDATE_EPOCHS
    ):

        np.random.shuffle(
            b_inds
        )

        for start in range(
            0,
            batch_size,
            minibatch_size
        ):

            end = (
                start
                + minibatch_size
            )

            mb_inds = b_inds[
                start:end
            ]

            _, newlogprob, entropy, newvalue = (
                agent.get_action_and_value(
                    b_obs[mb_inds],
                    b_actions[
                        mb_inds
                    ].long()
                )
            )

            logratio = (
                newlogprob
                - b_logprobs[
                    mb_inds
                ]
            )

            ratio = logratio.exp()

            mb_advantages = (
                b_advantages[
                    mb_inds
                ]
            )

            mb_advantages = (
                mb_advantages
                - mb_advantages.mean()
            ) / (
                mb_advantages.std()
                + 1e-8
            )

            # ----------------------------------------------------
            # POLICY LOSS
            # ----------------------------------------------------

            pg_loss1 = (
                -mb_advantages
                * ratio
            )

            pg_loss2 = (
                -mb_advantages
                * torch.clamp(
                    ratio,
                    1.0 - CLIP_COEF,
                    1.0 + CLIP_COEF
                )
            )

            pg_loss = torch.max(
                pg_loss1,
                pg_loss2
            ).mean()


            # ----------------------------------------------------
            # VALUE LOSS
            # ----------------------------------------------------

            newvalue = newvalue.view(-1)

            v_loss = 0.5 * (
                newvalue
                - b_returns[
                    mb_inds
                ]
            ).pow(2).mean()


            # ----------------------------------------------------
            # ENTROPY
            # ----------------------------------------------------

            entropy_loss = (
                entropy.mean()
            )


            # ----------------------------------------------------
            # TOTAL LOSS
            # ----------------------------------------------------

            loss = (
                pg_loss
                - ENT_COEF
                * entropy_loss
                + VF_COEF
                * v_loss
            )

            optimizer.zero_grad()

            loss.backward()

            nn.utils.clip_grad_norm_(
                agent.parameters(),
                MAX_GRAD_NORM
            )

            optimizer.step()


    # ------------------------------------------------------------
    # PROGRESS
    # ------------------------------------------------------------

    if (
        update % 5 == 0
        or update == 1
    ):

        elapsed = (
            time.time()
            - start_time
        )

        print(
            f"Update {update:3d}/{num_updates} | "
            f"Steps {global_step:6d}/{num_updates * batch_size} | "
            f"Loss {loss.item():.4f} | "
            f"Time {elapsed:.1f}s"
        )


# ================================================================
# 10. TRAINING COMPLETE
# ================================================================

print("\n" + "=" * 70)
print("TRAINING COMPLETE")
print("=" * 70)

print(
    f"Actual environment steps: {global_step}"
)


# ================================================================
# 11. SAVE MODEL IMMEDIATELY
# ================================================================

model_path = Path(
    "/content/ppo-LunarLander-v2.pt"
)

config_path = Path(
    "/content/ppo-LunarLander-v2-config.json"
)

torch.save(
    agent.state_dict(),
    model_path
)

config = {
    "env_id": ENV_ID,
    "total_timesteps": TOTAL_TIMESTEPS,
    "actual_timesteps": global_step,
    "num_envs": NUM_ENVS,
    "num_steps": NUM_STEPS,
    "learning_rate": LEARNING_RATE,
    "gamma": GAMMA,
    "gae_lambda": GAE_LAMBDA,
    "num_minibatches": NUM_MINIBATCHES,
    "update_epochs": UPDATE_EPOCHS,
    "clip_coef": CLIP_COEF,
    "ent_coef": ENT_COEF,
    "vf_coef": VF_COEF,
    "max_grad_norm": MAX_GRAD_NORM,
    "seed": SEED,
    "device": str(device)
}

with open(
    config_path,
    "w"
) as f:

    json.dump(
        config,
        f,
        indent=2
    )

print(
    f"✓ Model saved: {model_path}"
)

print(
    f"✓ Config saved: {config_path}"
)


# ================================================================
# 12. CLOSE TRAINING ENV
# ================================================================

envs.close()


# ================================================================
# 13. EVALUATION ENVIRONMENT
# ================================================================

eval_env = gym.make(
    ENV_ID
)


# ================================================================
# 14. EVALUATION
# ================================================================

def evaluate_agent(
    agent,
    env,
    episodes=10
):

    agent.eval()

    episode_rewards = []

    for episode in range(
        episodes
    ):

        obs = env.reset()

        if isinstance(obs, tuple):
            obs = obs[0]

        done = False

        episode_reward = 0.0

        while not done:

            obs_tensor = torch.tensor(
                obs,
                dtype=torch.float32,
                device=device
            ).unsqueeze(0)

            with torch.no_grad():

                action, _, _, _ = (
                    agent.get_action_and_value(
                        obs_tensor
                    )
                )

            action = (
                action
                .cpu()
                .numpy()[0]
            )

            result = env.step(
                int(action)
            )

            if len(result) == 5:

                obs, reward, terminated, truncated, info = result

                done = (
                    terminated
                    or truncated
                )

            else:

                obs, reward, done, info = result

            episode_reward += reward

        episode_rewards.append(
            episode_reward
        )

        print(
            f"Episode {episode + 1:2d}/{episodes}: "
            f"{episode_reward:.2f}"
        )

    agent.train()

    return (
        float(np.mean(episode_rewards)),
        float(np.std(episode_rewards))
    )


print("\n" + "=" * 70)
print("EVALUATING AGENT")
print("=" * 70)

mean_reward, std_reward = evaluate_agent(
    agent,
    eval_env,
    episodes=10
)

print(
    f"\nMean reward: {mean_reward:.2f}"
)

print(
    f"Std reward : {std_reward:.2f}"
)


# ================================================================
# 15. RECORD VIDEO
# ================================================================

print("\n" + "=" * 70)
print("RECORDING VIDEO")
print("=" * 70)

video_path = Path(
    "/content/replay.mp4"
)

video_env = gym.make(
    ENV_ID
)

frames = []

obs = video_env.reset()

if isinstance(obs, tuple):
    obs = obs[0]

done = False
video_steps = 0

while (
    not done
    and video_steps < 3000
):

    try:

        frame = video_env.render(
            mode="rgb_array"
        )

        if frame is not None:
            frames.append(frame)

    except Exception:
        pass

    obs_tensor = torch.tensor(
        obs,
        dtype=torch.float32,
        device=device
    ).unsqueeze(0)

    with torch.no_grad():

        action, _, _, _ = (
            agent.get_action_and_value(
                obs_tensor
            )
        )

    action = (
        action
        .cpu()
        .numpy()[0]
    )

    result = video_env.step(
        int(action)
    )

    if len(result) == 5:

        obs, reward, terminated, truncated, info = result

        done = (
            terminated
            or truncated
        )

    else:

        obs, reward, done, info = result

    video_steps += 1

video_env.close()

if frames:

    imageio.mimsave(
        str(video_path),
        frames,
        fps=30
    )

    print(
        f"✓ Video saved: {video_path}"
    )

else:

    print(
        "⚠ Video frames were not captured."
    )


# ================================================================
# 16. PREPARE HF PACKAGE
# ================================================================

print("\n" + "=" * 70)
print("PREPARING HUGGING FACE PACKAGE")
print("=" * 70)

package_dir = Path(
    "/content/ppo-LunarLander-v2-package"
)

package_dir.mkdir(
    parents=True,
    exist_ok=True
)

shutil.copy(
    model_path,
    package_dir / model_path.name
)

shutil.copy(
    config_path,
    package_dir / config_path.name
)

if video_path.exists():

    shutil.copy(
        video_path,
        package_dir / "replay.mp4"
    )


# ================================================================
# 17. EVALUATION JSON
# ================================================================

evaluation = {
    "environment": ENV_ID,
    "mean_reward": mean_reward,
    "std_reward": std_reward,
    "episodes": 10,
    "evaluation_date": datetime.datetime.now().isoformat()
}

with open(
    package_dir / "evaluation.json",
    "w"
) as f:

    json.dump(
        evaluation,
        f,
        indent=2
    )


# ================================================================
# 18. README / MODEL CARD
# ================================================================

readme = f"""---
tags:
- deep-reinforcement-learning
- reinforcement-learning
- lunarlander
- pytorch
- unit8
library_name: pytorch
---

# PPO LunarLander-v2

A Proximal Policy Optimization (PPO) agent
trained from scratch using PyTorch.

## Environment

- Environment: LunarLander-v2
- Algorithm: PPO
- Framework: PyTorch
- Parallel environments: {NUM_ENVS}
- Steps per rollout: {NUM_STEPS}
- Total timesteps: {TOTAL_TIMESTEPS}
- Actual timesteps: {global_step}

## Hyperparameters

- Learning rate: {LEARNING_RATE}
- Gamma: {GAMMA}
- GAE lambda: {GAE_LAMBDA}
- PPO clip coefficient: {CLIP_COEF}
- Entropy coefficient: {ENT_COEF}
- Value function coefficient: {VF_COEF}

## Evaluation

Mean reward: {mean_reward:.2f}

Standard deviation: {std_reward:.2f}

## Files

- `ppo-LunarLander-v2.pt`
- `ppo-LunarLander-v2-config.json`
- `evaluation.json`
- `replay.mp4`

## Hugging Face Deep Reinforcement Learning Course

This model was trained as part of Unit 8
of the Hugging Face Deep Reinforcement Learning Course.
"""

with open(
    package_dir / "README.md",
    "w"
) as f:

    f.write(readme)

print("✓ README created")


# ================================================================
# 19. CREATE / UPDATE HF REPOSITORY
# ================================================================

print("\n" + "=" * 70)
print("UPLOADING TO HUGGING FACE HUB")
print("=" * 70)

api = HfApi()

api.create_repo(
    repo_id=REPO_ID,
    repo_type="model",
    exist_ok=True
)

print(
    f"✓ Repository ready: {REPO_ID}"
)


# ================================================================
# 20. UPLOAD
# ================================================================

upload_folder(
    folder_path=str(package_dir),
    repo_id=REPO_ID,
    repo_type="model",
    commit_message="Upload PPO LunarLander-v2 trained model"
)


# ================================================================
# 21. FINAL RESULT
# ================================================================

print("\n" + "=" * 70)
print("🎉 UNIT 8 PPO COMPLETE 🎉")
print("=" * 70)

print(
    f"\nHugging Face repository:"
)

print(
    f"https://huggingface.co/{REPO_ID}"
)

print(
    f"\nMean evaluation reward: {mean_reward:.2f}"
)

print(
    f"Std evaluation reward : {std_reward:.2f}"
)

print("\nUploaded files:")

for file in sorted(
    package_dir.iterdir()
):

    print(
        f"  ✓ {file.name}"
    )

print(
    "\n✅ Training + evaluation + video + packaging + Hub upload finished!"
)

In [ ]:
from huggingface_hub import notebook_login
notebook_login()
!git config --global credential.helper store

If you don't want to use a Google Colab or a Jupyter Notebook, you need to use this command instead: `huggingface-cli login`

## Let's start the training 🔥
- ⚠️ ⚠️ ⚠️  Don't use **the same repo id with the one you used for the Unit 1**
- Now that you've coded from scratch PPO and added the Hugging Face Integration, we're ready to start the training 🔥

- First, you need to copy all your code to a file you create called `ppo.py`

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit9/step1.png" alt="PPO"/>

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit9/step2.png" alt="PPO"/>

- Now we just need to run this python script using `python <name-of-python-script>.py` with the additional parameters we defined with `argparse`

- You should modify more hyperparameters otherwise the training will not be super stable.

In [ ]:
!python ppo.py --env-id="LunarLander-v2" --repo-id="YOUR_REPO_ID" --total-timesteps=50000

## Some additional challenges 🏆
The best way to learn **is to try things by your own**! Why not trying  another environment?


See you on Unit 8, part 2 where we going to train agents to play Doom 🔥
## Keep learning, stay awesome 🤗